# Coalitions-02 — Assistance Games (résultat 2026)

**Jumeau C#** — kernel `.net-csharp`, BCL seule (aucun NuGet, aucun accès réseau).

**Source du portage** : [Coalitions-02-Assistance-Games-Python.ipynb](Coalitions-02-Assistance-Games-Python.ipynb)
**Navigation** : [<< 15-CooperativeGames (track principal)](../GameTheory-15-CooperativeGames-Python.ipynb) · [Index de la sous-série](README.md) · [Index de la série](../README.md) · [Coalitions-03 >>](Coalitions-03-Echange-de-Reins-Lean-Python.ipynb)

**Side tracks 15x** : [15b-Lean-CooperativeGames](../GameTheory-15b-CooperativeGames-Lean.ipynb) · [15c-CooperativeGames-Python](../GameTheory-15c-CooperativeGames-Python.ipynb) · [SC-06b-Mobius-Coalitions](../SocialChoice/ChoixSocial-06b-Mobius-Coalitions-Lean-Python.ipynb) · [Coalitions-01-Pouvoir-Coalitionnel](Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb) · [15d-Shapley-Groupes](../GameTheory-15d-Shapley-Groupes-Python.ipynb)

**Ce que ce jumeau rejoue** : la même instance jouet (deux préférences latentes, deux signaux humains, trois actions assistant), le même oracle de politique jointe en hindsight, le même apprenant EXP3, le même banc de regret sur quatre horizons, et la même lecture du résultat — recalculés en BCL.

**Conventions de parité** : les séquences de préférences `θ(t)` et les valeurs de référence du jumeau Python sont reprises comme **données citées** (graine 42, une séquence par horizon) ; elles ne sont pas recalculées ici. Les valeurs que ce carnet produit — oracle, regret, écart-type, exposant log-log — sont, elles, recalculées et vérifiées par des assertions exécutables.

**Papier** : *Provably Optimal Learning Algorithms for Assistance Games* — Ananthakrishnan, Bedaywi, Jordan, Russell, Haghtalab (UC Berkeley), 2026. arXiv 2607.08012.


In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Jumeau C# du carnet Coalitions-02-Assistance-Games : la meme instance jouet, le meme
// oracle en hindsight, le meme apprenant EXP3, le meme banc de regret -- recalcules en BCL.
//
// Le module Python `cooperative_games.assistance_games` (fonction `off_switch_game`,
// `override_threshold = 0.9`) reste CITE comme provenance : rien n'en est importe ici,
// les seules fonctions consommees par le carnet sont reimplementees en BCL.

Console.WriteLine("Notebook Coalitions-02 - Assistance Games 2026 charge");
Console.WriteLine("Organe natif cooperative_games.assistance_games cite comme provenance ; reimplementation BCL dans ce carnet");


Notebook Coalitions-02 - Assistance Games 2026 charge


Organe natif cooperative_games.assistance_games cite comme provenance ; reimplementation BCL dans ce carnet


### 3.1 Modèle de jeu (Θ, A_H, A_A, r) — instance jouet

Pour T=20 tours, |Θ|=2 préférences latentes, |A_H|=2 signaux humains, |A_A|=3 actions assistant. La récompense `r(a_H, a_A, θ)` est commune et vaut au plus 1.

Le module Python `cooperative_games.assistance_games` (fonction `off_switch_game`, `override_threshold = 0.9`) reste **cité comme provenance** : ce jumeau n'en importe rien, les fonctions consommées par le carnet sont réimplémentées en BCL.


In [2]:
// Instance jouet du modele (Theta, A_H, A_A, r) : deux preferences latentes,
// deux signaux humains, trois actions assistant.
// r ne depend pas de a_H dans cette instance : l'action humaine est un signal,
// la decision appartient a l'assistant.

var THETAS = new double[] { 0.3, 0.7 };
var A_H = new string[] { "signal_paperclip", "signal_staple" };
var A_A = new string[] { "90_paperclips", "90_staples", "50_each" };

double Reward(int thetaIdx, int aHIdx, int aAIdx)
{
    double theta = THETAS[thetaIdx];
    if (A_A[aAIdx] == "90_paperclips")
    {
        return 90.0 * theta / 90.0;
    }
    else if (A_A[aAIdx] == "90_staples")
    {
        return 90.0 * (1.0 - theta) / 90.0;
    }
    else
    {
        return 50.0 * (theta + (1.0 - theta)) / 90.0;
    }
}

// Assertions executables : les valeurs de l'instance sont verifiees, pas seulement imprimees.
void Verifie(bool condition, string message)
{
    if (!condition)
    {
        throw new InvalidOperationException(message);
    }
}

Verifie(THETAS.Length == 2, "|Theta| doit valoir 2");
Verifie(A_H.Length == 2, "|A_H| doit valoir 2");
Verifie(A_A.Length == 3, "|A_A| doit valoir 3");
Verifie(Math.Abs(Reward(0, 0, 0) - 0.3) < 1e-12, "r(theta=0.3, 90_paperclips) doit valoir 0.3");
Verifie(Math.Abs(Reward(0, 0, 1) - 0.7) < 1e-12, "r(theta=0.3, 90_staples) doit valoir 0.7");
Verifie(Math.Abs(Reward(1, 0, 0) - 0.7) < 1e-12, "r(theta=0.7, 90_paperclips) doit valoir 0.7");
Verifie(Math.Abs(Reward(0, 0, 2) - 50.0 / 90.0) < 1e-12, "r(theta, 50_each) doit valoir 50/90");
Verifie(Math.Abs(Reward(1, 0, 2) - 50.0 / 90.0) < 1e-12, "r(theta, 50_each) doit valoir 50/90");
Verifie(Reward(0, 0, 0) == Reward(0, 1, 0), "r ne doit pas dependre de a_H dans cette instance");

Console.WriteLine("Toy instance : |Theta|=" + THETAS.Length + ", |A_H|=" + A_H.Length + ", |A_A|=" + A_A.Length);
Console.WriteLine("Thetaprint : thetas, A_H, A_A, reward OK");


Toy instance : |Theta|=2, |A_H|=2, |A_A|=3


Thetaprint : thetas, A_H, A_A, reward OK


### 3.2 Politique jointe optimale en hindsight (oracle)

Pour chaque tour, l'oracle connaît `θ(t)` et choisit la paire `(a_H, a_A)` qui maximise la récompense cumulée. C'est la borne que l'assistant doit approcher : tout regret se lit comme l'écart à cette borne.


In [3]:
// Politique jointe optimale en hindsight : a chaque tour, l'oracle connait theta(t)
// et choisit la meilleure paire (a_H, a_A).

double OptimalJointPolicyHindsight(double[] thetas, string[] aH, string[] aA, Func<int, int, int, double> reward, int[] thetaSequence)
{
    double cumulative = 0.0;
    foreach (int thetaIdx in thetaSequence)
    {
        double best = -1.0;
        for (int ih = 0; ih < aH.Length; ih++)
        {
            for (int ia = 0; ia < aA.Length; ia++)
            {
                double r = reward(thetaIdx, ih, ia);
                if (r > best)
                {
                    best = r;
                }
            }
        }
        cumulative += best;
    }
    return cumulative;
}

// Sequence theta(t) du jumeau Python (graine 42, T = 20) : DONNEE CITEE,
// reprise telle quelle, jamais recalculee ici.
int[] thetaSequence20 = new int[] { 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0 };

double optimalCum = OptimalJointPolicyHindsight(THETAS, A_H, A_A, Reward, thetaSequence20);
Console.WriteLine("T=20, optimal cumulative reward (hindsight oracle) = " + optimalCum.ToString("F3", CultureInfo.InvariantCulture));

// Valeur CITEE du jumeau Python pour la meme instance et la meme sequence : 14.000.
Verifie(Math.Abs(optimalCum - 14.000) < 1e-9, "l'oracle doit retrouver 14.000 (valeur citee du jumeau Python)");


T=20, optimal cumulative reward (hindsight oracle) = 14.000


### 3.3 Algorithme EXP3 pour l'assistant (apprenant sans regret)

EXP3 (Auer et al., 2002) est un algorithme sans regret pour le bandit manchot. Adapté ici à l'assistant, qui choisit `a_A ∈ A_A` **sans observer θ** : il ne voit que le signal `a_H`.

Borne standard : regret `O(√(K T log K))` pour K actions et T tours.

**Le point honnête de cette implémentation** : `select_action` ne reçoit pas `a_H` — la distribution sur `A_A` est donc identique à chaque tour. Sur l'instance jouet 2×2×3, le regret est dominé par le plancher de la meilleure politique fixe, indépendamment de γ. C'est une limite du learner non conditionnel câblé ici, pas d'EXP3 lui-même ; la parade attendue par le papier est une politique conditionnelle `π_A(a_H)`.


In [4]:
// EXP3 (Auer et al., 2002) : apprenant sans regret sur un bandit manchot.
// L'assistant est SIGNAL-AVEUGLE : sa distribution sur A_A ne depend pas de a_H.
// C'est une limite de cette instance jouet, pas d'EXP3 lui-meme : le papier attend
// une politique conditionnelle pi_A(a_H).

class Exp3Assistant
{
    public int NActions;
    public double Gamma;
    public double[] Weights;
    private readonly Random _rng;

    public Exp3Assistant(int nActions, int tHorizon, Random rng)
    {
        NActions = nActions;
        _rng = rng;
        Gamma = Math.Min(1.0, Math.Sqrt(nActions * Math.Log(Math.Max(2, nActions)) / ((Math.E - 1.0) * Math.Max(1, tHorizon))));
        Weights = new double[nActions];
        for (int i = 0; i < nActions; i++)
        {
            Weights[i] = 1.0;
        }
    }

    public double[] Probs()
    {
        double sum = Weights.Sum();
        var probs = new double[NActions];
        for (int i = 0; i < NActions; i++)
        {
            probs[i] = (1.0 - Gamma) * Weights[i] / sum + Gamma / NActions;
        }
        return probs;
    }

    public int SelectAction()
    {
        double[] probs = Probs();
        double u = _rng.NextDouble();
        double acc = 0.0;
        for (int i = 0; i < NActions; i++)
        {
            acc += probs[i];
            if (u < acc)
            {
                return i;
            }
        }
        return NActions - 1;
    }

    public void Update(int action, double reward)
    {
        double[] probs = Probs();
        double estimatedReward = reward / probs[action];
        Weights[action] *= Math.Exp(Gamma * estimatedReward / NActions);
    }
}

// Simulation : l'humain joue une politique honnete (son signal est l'index de theta),
// l'assistant joue EXP3. Meme graines (seed * 1000 + 42) que la source.

(double Mean, double Std) SimulateExp3(int[] thetaSequence, int nSeeds)
{
    int T = thetaSequence.Length;
    int nA = A_A.Length;
    var cumulatives = new double[nSeeds];
    for (int seed = 0; seed < nSeeds; seed++)
    {
        var rng = new Random(seed * 1000 + 42);
        var assistant = new Exp3Assistant(nA, T, rng);
        double cumul = 0.0;
        for (int t = 0; t < T; t++)
        {
            int thetaIdx = thetaSequence[t];
            int aHIdx = thetaIdx;                 // signal honnete = index theta
            int aAIdx = assistant.SelectAction();
            double r = Reward(thetaIdx, aHIdx, aAIdx);
            assistant.Update(aAIdx, r);
            cumul += r;
        }
        cumulatives[seed] = cumul;
    }
    double mean = cumulatives.Average();
    double std = Math.Sqrt(cumulatives.Sum(v => (v - mean) * (v - mean)) / nSeeds);
    return (mean, std);
}

var (meanR, stdR) = SimulateExp3(thetaSequence20, 10);

// Valeurs CITEES du jumeau Python (meme instance, meme sequence, dix graines).
double citedMean = 10.218;
double citedStd = 0.661;
double sem = citedStd / Math.Sqrt(10.0);          // erreur standard de la moyenne sur dix graines
double tolerance = 3.0 * sem;                     // bande de trois sigma declaree

Console.WriteLine("EXP3 (signal-aveugle) moyen cumule sur 10 seeds = " + meanR.ToString("F3", CultureInfo.InvariantCulture) + " +/- " + stdR.ToString("F3", CultureInfo.InvariantCulture));
Console.WriteLine("Oracle cumule = " + optimalCum.ToString("F3", CultureInfo.InvariantCulture));

double regret = optimalCum - meanR;
Console.WriteLine("Assistance regret empirique = " + regret.ToString("F3", CultureInfo.InvariantCulture) + " (" + (regret / optimalCum).ToString("F3", CultureInfo.InvariantCulture) + " relatif)");

double ecart = Math.Abs(meanR - citedMean);
Console.WriteLine("Ecart au cite Python = " + ecart.ToString("F3", CultureInfo.InvariantCulture) + " (tolerance declaree 3 sigma = " + tolerance.ToString("F3", CultureInfo.InvariantCulture) + ")");
Verifie(ecart <= tolerance, "le regret mesure doit rester dans la bande de trois sigma autour de la valeur citee");

// Plancher structurel : un apprenant qui n'observe pas theta ne peut pas depasser
// la meilleure action independante de theta (50_each = 50/90 par tour).
double meilleureActionFixe = 50.0 / 90.0;
double moyenneParTour = meanR / 20.0;
double marge = 3.0 * (stdR / Math.Sqrt(10.0)) / 20.0;
Verifie(moyenneParTour <= meilleureActionFixe + marge, "un apprenant signal-aveugle ne peut pas depasser la meilleure action independante de theta");

Console.WriteLine("Note : regret domine par le plancher signal-aveugle (voir la lecture du banc).");


EXP3 (signal-aveugle) moyen cumule sur 10 seeds = 10.486 +/- 0.685


Oracle cumule = 14.000


Assistance regret empirique = 3.514 (0.251 relatif)


Ecart au cite Python = 0.268 (tolerance declaree 3 sigma = 0.627)


Note : regret domine par le plancher signal-aveugle (voir la lecture du banc).


### 3.4 Banc de regret — vérification empirique sur quatre horizons

Le papier distingue deux régimes de borne d'assistance regret : le régime pseudo-décentralisé (chaîne aléatoire partagée) en `Õ(√T)`, et le régime décentralisé pur en `Õ(T^(3/4))`.

Ce banc mesure le regret d'assistance empirique pour `T ∈ {10, 20, 40, 80}`, dix graines par horizon, et compare le profil obtenu aux deux courbes de référence.

Les séquences `θ(t)` sont les **données citées** du jumeau Python (graine 42, une graine par horizon) : l'oracle est donc recalculé exactement ici, sur la même instance ; seuls les tirages de l'apprenant diffèrent, d'où la bande de comparaison déclarée (3 σ de l'estimateur de moyenne sur dix graines).


In [5]:
// Banc : T ∈ {10, 20, 40, 80}. Les sequences theta(t) sont les DONNEES CITEES du jumeau
// Python (graine 42, une graine par T) : l'oracle est donc recalcule exactement ici,
// sur la meme instance ; seuls les tirages de l'apprenant different.

int[] tValues = new int[] { 10, 20, 40, 80 };

var thetaSeqCite = new Dictionary<int, int[]>
{
    { 10, new int[] { 0, 1, 1, 0, 0, 1, 0, 1, 0, 0 } },
    { 20, new int[] { 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0 } },
    { 40, new int[] { 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1 } },
    { 80, new int[] { 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1 } }
};

// Regrets CITES du jumeau Python (moyenne sur dix graines) et ecarts-types cites.
var citedRegrets = new Dictionary<int, double> { { 10, 1.960 }, { 20, 3.782 }, { 40, 7.426 }, { 80, 14.140 } };
var citedStds = new Dictionary<int, double> { { 10, 0.487 }, { 20, 0.661 }, { 40, 1.212 }, { 80, 1.839 } };

var oracles = new Dictionary<int, double>();
var regretsMesures = new Dictionary<int, double>();
var stdsMesures = new Dictionary<int, double>();

foreach (int T in tValues)
{
    int[] seq = thetaSeqCite[T];
    double oracle = OptimalJointPolicyHindsight(THETAS, A_H, A_A, Reward, seq);
    var (m, s) = SimulateExp3(seq, 10);
    double r = oracle - m;
    oracles[T] = oracle;
    regretsMesures[T] = r;
    stdsMesures[T] = s;

    // L'oracle est exact : meme sequence citee, meme instance, meme calcul.
    Verifie(Math.Abs(oracle - 0.7 * T) < 1e-9, "l'oracle doit valoir 0.7 par tour sur la sequence citee");

    double tol = 3.0 * citedStds[T] / Math.Sqrt(10.0);
    double ecart = Math.Abs(r - citedRegrets[T]);
    Verifie(ecart <= tol, "le regret mesure doit rester dans la bande de trois sigma autour de la valeur citee");

    Console.WriteLine("T=" + T.ToString().PadLeft(3) + " | regret empirique = " + r.ToString("F3", CultureInfo.InvariantCulture) + " +/- " + s.ToString("F3", CultureInfo.InvariantCulture));
}

// Exposant mesure : regression log-log sur T ∈ {20, 40, 80}.
double ExposantLogLog(double[] xs, double[] ys)
{
    double mx = xs.Select(v => Math.Log(v)).Average();
    double my = ys.Select(v => Math.Log(v)).Average();
    double num = 0.0;
    double den = 0.0;
    for (int i = 0; i < xs.Length; i++)
    {
        double dx = Math.Log(xs[i]) - mx;
        double dy = Math.Log(ys[i]) - my;
        num += dx * dy;
        den += dx * dx;
    }
    return num / den;
}

double[] xs = new double[] { 20.0, 40.0, 80.0 };
double[] ysCite = new double[] { citedRegrets[20], citedRegrets[40], citedRegrets[80] };
double[] ysMesures = new double[] { regretsMesures[20], regretsMesures[40], regretsMesures[80] };

double alphaCite = ExposantLogLog(xs, ysCite);
double alphaMesure = ExposantLogLog(xs, ysMesures);

Verifie(Math.Abs(alphaCite - 0.95) < 0.02, "l'exposant log-log des valeurs citees doit valoir 0.95");
Verifie(alphaMesure > 0.70 && alphaMesure < 1.15, "le regime mesure doit rester quasi lineaire (exposant entre 0.70 et 1.15)");

Console.WriteLine("Exposant log-log sur T ∈ {20, 40, 80} : cite (Python) = " + alphaCite.ToString("F2", CultureInfo.InvariantCulture) + " | mesure (C#) = " + alphaMesure.ToString("F2", CultureInfo.InvariantCulture));

// Rendu texte : les deux courbes de reference sont affichees en regard. Elles decrivent
// d'autres algorithmes (regimes pseudo-decentralise et decentralise) : le regret mesure,
// sur une instance jouet et un apprenant signal-aveugle, ne leur est pas oppose.
Console.WriteLine();
Console.WriteLine("   T | oracle  | regret mesure +/- ecart-type | regret/tour | cite (Python) | 0.3*sqrt(T) | 0.3*T^(3/4)");
foreach (int T in tValues)
{
    Console.WriteLine(
        T.ToString().PadLeft(4) + " | " + oracles[T].ToString("F3", CultureInfo.InvariantCulture).PadLeft(7) + " | "
        + regretsMesures[T].ToString("F3", CultureInfo.InvariantCulture).PadLeft(8) + " +/- "
        + stdsMesures[T].ToString("F3", CultureInfo.InvariantCulture).PadLeft(5) + " | "
        + (regretsMesures[T] / T).ToString("F3", CultureInfo.InvariantCulture).PadLeft(10) + " | "
        + citedRegrets[T].ToString("F3", CultureInfo.InvariantCulture).PadLeft(13) + " | "
        + (0.3 * Math.Sqrt(T)).ToString("F3", CultureInfo.InvariantCulture).PadLeft(11) + " | "
        + (0.3 * Math.Pow(T, 0.75)).ToString("F3", CultureInfo.InvariantCulture).PadLeft(11));
}
Console.WriteLine();
Console.WriteLine("Bench termine, rendu texte (le jumeau C# ne produit pas de figure PNG).");


T= 10 | regret empirique = 1.776 +/- 0.663


T= 20 | regret empirique = 3.514 +/- 0.685


T= 40 | regret empirique = 7.537 +/- 1.055


T= 80 | regret empirique = 15.158 +/- 1.225


Exposant log-log sur T ∈ {20, 40, 80} : cite (Python) = 0.95 | mesure (C#) = 1.05


   T | oracle  | regret mesure +/- ecart-type | regret/tour | cite (Python) | 0.3*sqrt(T) | 0.3*T^(3/4)


  10 |   7.000 |    1.776 +/- 0.663 |      0.178 |         1.960 |       0.949 |       1.687


  20 |  14.000 |    3.514 +/- 0.685 |      0.176 |         3.782 |       1.342 |       2.837


  40 |  28.000 |    7.537 +/- 1.055 |      0.188 |         7.426 |       1.897 |       4.772


  80 |  56.000 |   15.158 +/- 1.225 |      0.189 |        14.140 |       2.683 |       8.025


Bench termine, rendu texte (le jumeau C# ne produit pas de figure PNG).


### 3.5 Lecture du résultat

**Ce que le jumeau C# confirme, valeur par valeur** : l'oracle en hindsight vaut `0.7` de récompense par tour sur cette instance — quelles que soient les préférences — donc `14.000` sur vingt tours ; l'apprenant EXP3 signal-aveugle plafonne autour de `0.5` de récompense par tour ; l'écart entre les deux, normalisé, reste autour de `0.18` quel que soit l'horizon.

**Ce qui ne change pas d'un jumeau à l'autre** : le regret mesuré est dominé par le **plancher de la meilleure politique fixe**. Comme la distribution de l'assistant ne dépend pas du signal humain, celui-ci ne peut pas faire mieux que l'action indépendante de `θ` (`50_each`, soit `50/90` par tour) ; le regret normalisé reste donc borné par le plancher, et croît de façon quasi linéaire sur cet intervalle d'horizons. La courbe de référence `√T` et la courbe `T^(3/4)` du papier décrivent **d'autres algorithmes** — le regret mesuré ici ne leur est pas opposé.

**Ce qui dépend du générateur aléatoire** : la moyenne de l'apprenant vaut `10.486 +/- 0.685` dans ce jumeau, contre `10.218 +/- 0.661` dans le jumeau Python. L'écart reste dans la bande de trois sigma déclarée, mais il ne s'annule pas : les flux pseudo-aléatoires des deux langages diffèrent, et seule la comparaison à tolérance peut établir la parité — pas l'égalité des tirages.

**Ce que ce jumeau n'établit pas** : la borne `(1 - 1/e)` du théorème, ni la distinction asymptotique entre les deux régimes. Une instance plus grande, et un apprenant qui observe `a_H`, seraient nécessaires — c'est la limite méthodologique que le carnet assume, et elle est la même des deux côtés.


In [6]:
// Exercice 1 : politique humaine non-honnete.
// TODO etudiant : simuler le jeu d'assistance avec un humain qui ment sur sa preference,
// puis comparer le regret cumule de l'assistant au cas honnete.
double SimulateDeceptiveHuman(double[] thetas, string[] aH, string[] aA, Func<int, int, int, double> reward, int[] thetaSequence, int rngSeed)
{
    // TODO etudiant
    return 0.0;
}

Console.WriteLine("Exercice a completer : le regret de l'assistant augmente-t-il quand le signal humain ment ?");


Exercice a completer : le regret de l'assistant augmente-t-il quand le signal humain ment ?


In [7]:
// Exercice 2 : regret cumule contre oracle en fonction de T (trace log-log).
// TODO etudiant : renvoyer la liste (T, regret moyen) pour chaque horizon demande.
List<(int T, double Regret)> ComputeRegretScaling(double[] thetas, string[] aH, string[] aA, Func<int, int, int, double> reward, int[] tValues, int nSeeds)
{
    // TODO etudiant
    return new List<(int T, double Regret)>();
}

Console.WriteLine("Exercice a completer : la courbe suit-elle sqrt(T) ou T^(3/4) ?");


Exercice a completer : la courbe suit-elle sqrt(T) ou T^(3/4) ?


In [8]:
// Exercice 3 : etendre a |Theta| = 3 preferences (cas general du papier).
// TODO etudiant : construire l'instance a trois preferences latentes et la rendre avec
// (preferences, actions humaines, actions assistant, fonction de recompense).
(double[] Thetas, string[] AH, string[] AA, Func<int, int, int, double> Reward3) Build3PrefInstance()
{
    // TODO etudiant
    return (Array.Empty<double>(), Array.Empty<string>(), Array.Empty<string>(), (t, h, a) => 0.0);
}

Console.WriteLine("Exercice a completer : l'EXP3 s'adapte-t-il a trois preferences latentes ?");


Exercice a completer : l'EXP3 s'adapte-t-il a trois preferences latentes ?


## Conclusion

Ce jumeau C# rejoue sur l'instance jouet le même protocole que son carnet Python : l'oracle de politique jointe en hindsight, l'apprenant EXP3 signal-aveugle, le banc de regret sur quatre horizons, et la lecture qui en découle.

**Résultat empirique** : un régime quasi linéaire à cet horizon (exposant log-log proche de 0.95 sur les valeurs citées), un regret dominé par le plancher de la meilleure politique fixe, et un écart à l'oracle qui se lit en valeur absolue et en valeur relative.

**Leçon méthodologique** : le piège principal n'est pas γ, c'est le **signal-aveugle du learner**. Une distribution d'actions qui ne consomme pas `a_H` applique la même loi à chaque tour ; le regret cumulé mesure alors le prix de cet aveuglement, non une défaillance d'EXP3. La parade (une politique conditionnelle `π_A(a_H)`) reste la marche suivante.

**Sources** :

- *Provably Optimal Learning Algorithms for Assistance Games*, Ananthakrishnan, Bedaywi, Jordan, Russell, Haghtalab (2026), arXiv `2607.08012` — archivé dans `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\Cooperation, Institutions and LLM Agents\2026 - Ananthakrishnan et al - Provably Optimal Learning Algorithms for Assistance Games (arXiv 2607.08012).pdf`.
- Auer, P., Cesa-Bianchi, N., Freund, Y., & Schapire, R. E. (2002). *The nonstochastic multiarmed bandit problem*. SIAM Journal on Computing.
- Hadfield-Menell et al. (2017). *The Off-Switch Game*. arXiv `1611.08219` — cité par [GameTheory-15-CooperativeGames-Python.ipynb](../GameTheory-15-CooperativeGames-Python.ipynb).


## À retenir

| Leçon | Référence citée (jumeau Python) | Valeur mesurée dans ce jumeau C# |
|---|---|---|
| L'oracle en hindsight borne la récompense atteignable sur cette instance | `T=20, optimal cumulative reward (hindsight oracle) = 14.000` | `T=20, optimal cumulative reward (hindsight oracle) = 14.000` |
| L'apprenant EXP3 de ce carnet est **signal-aveugle** : il n'observe pas l'action humaine | `EXP3 (signal-aveugle) moyen cumulé sur 10 seeds = 10.218 +/- 0.661` | `EXP3 (signal-aveugle) moyen cumulé sur 10 seeds = 10.486 +/- 0.685` |
| Le regret est l'écart à l'oracle, lu en valeur absolue et en valeur relative | `Assistance regret empirique = 3.782 (0.270 relatif)` | `Assistance regret empirique = 3.514 (0.251 relatif)` |
| L'oracle recalculé au fil de la simulation retrouve la borne de la section 3.2 | `Oracle cumulé = 14.000` | `Oracle cumulé = 14.000` |
| Le profil du regret reste quasi linéaire sur l'intervalle d'horizons balayé | `0.95` (lu dans la lecture du jumeau Python) | `Exposant log-log sur T ∈ {20, 40, 80} : cite (Python) = 0.95 \| mesure (C#) = 1.05` |

L'oracle est **exactement** identique des deux côtés : la séquence de préférences citée et l'instance de calcul sont les mêmes. Les valeurs de l'apprenant diffèrent légèrement, parce que le générateur pseudo-aléatoire diffère ; l'écart est affiché et borné par la bande de trois sigma déclarée, et chaque comparaison est portée par une assertion exécutable.


## Vérifiez votre compréhension

**Question 1 — Le regret mesuré est-il une défaillance de l'algorithme EXP3 ?**

**Réponse** : non. L'implémentation applique la même distribution sur `A_A` quel que soit l'action humaine ; le regret mesuré est donc dominé par le plancher de la meilleure politique fixe, pas par le paramètre de l'algorithme. Une politique conditionnelle `π_A(a_H)` est la parade, et elle n'est pas câblée ici.

**Question 2 — Que prouve la comparaison avec l'oracle en hindsight ?**

**Réponse** : que la valeur de l'oracle est la récompense cumulée qu'une politique jointe omnisciente aurait obtenue sur les tours de cette instance. Cela ne dit rien des deux bornes asymptotiques du théorème : les courbes de référence tracées en regard (`0.3 * sqrt(T)` et `0.3 * T^(3/4)`) décrivent d'autres algorithmes, et le profil mesuré se lit comme un régime quasi linéaire à cet horizon.

**Question 3 — Pourquoi les séquences de préférences ne sont-elles pas recalculées ici ?**

**Réponse** : parce qu'elles viennent d'un générateur pseudo-aléatoire dont le flux n'est pas reproductible d'une language à l'autre. Les reprendre comme données citées — et afficher l'écart entre les deux côtés — rend la parité vérifiable sans prétendre à une égalité qui n'existe pas.


## Pour aller plus loin

- [Index de la sous-série Coalitions](README.md) — le fil de la sous-série et ce que chaque carnet y ajoute.
- [Index de la série GameTheory](../README.md) — la place de l'assistance games parmi les autres structures de la théorie des jeux.
- [Coalitions-01 — Pouvoir coalitionnel](Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb) — le pouvoir de faire basculer une décision, encodé en SMT.
- [Coalitions-03 — Échange de reins](Coalitions-03-Echange-de-Reins-Lean-Python.ipynb) — de l'institution au graphe de compatibilité.
- [Coalitions-04 — Affectation de Kuhn–Munkres](Coalitions-04-Munkres-Affectation-Python.ipynb) — la méthode hongroise et son certificat.

Les trois exercices qui suivent la section 3.5 restent à faire : la politique humaine non honnête, la trace du regret cumulé contre l'oracle en fonction de `T`, et l'extension à trois préférences latentes — le cas général du papier, que ce carnet ne couvre pas.


## References

* Ananthakrishnan, N., Bedaywi, G., Jordan, M. I., Russell, S. J. & Haghtalab, S. (2026). *Provably Optimal Learning Algorithms for Assistance Games*. arXiv `2607.08012`. -> §3.1 (modèle de jeu (Θ, A_H, A_A, r)), §3.2 (politique jointe optimale en hindsight), §3.3 (algorithme EXP3 pour l'assistant), §3.4 (banc de vérification empirique), §3.5 (lecture du résultat), §5 (conclusion et limites méthodologiques) — la référence centrale du carnet, papier archivé dans `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\Cooperation, Institutions and LLM Agents\2026 - Ananthakrishnan et al - Provably Optimal Learning Algorithms for Assistance Games (arXiv 2607.08012).pdf`.

* Auer, P., Cesa-Bianchi, N., Freund, Y. & Schapire, R. E. (2002). *The Nonstochastic Multiarmed Bandit Problem*. SIAM Journal on Computing 32(1): 48-77. -> §3.3 (l'algorithme EXP3 fondateur, apprenant sans regret), §3.4 (la borne en racine de T qui définit le régime quasi linéaire observé à horizon fini), §5 (confrontation de la borne théorique au regret empirique).

* Hadfield-Menell, D., Dragan, A., Abbeel, P. & Russell, S. J. (2017). *The Off-Switch Game*. arXiv `1611.08219`. -> §3.1 (modèle de jeu avec signal — la politique conditionnelle π_A(a_H) que ce carnet n'implémente pas), §5 (leçon méthodologique sur le piège du signal-aveugle sur cette instance jouet 2x2x3).

* Cesa-Bianchi, N. & Lugosi, G. (2006). *Prediction, Learning, and Games*. Cambridge University Press. -> §3.3 (cadre général de l'apprentissage sans regret, définition et bornes), §3.4 (borne en racine de T utilisée comme référence de comparaison).

* Lattimore, T. & Szepesvari, C. (2020). *Bandit Algorithms*. Cambridge University Press. -> §3.4 (extension moderne du résultat Auer 2002), §5 (régime asymptotique, traité au ch. 9-11).

* Bartok, G., Foster, D. P., Pal, D., Rakhlin, A. & Szepesvari, C. (2014). *Partial Monitoring — Classification, Regret Bounds, and Algorithms*. Mathematics of Operations Research 39(4): 967-997. -> §3.1 (le modèle d'assistance games est un cas particulier de monitoring partiel : l'assistant observe un signal sans connaître la perte exacte), §5 (extensions naturelles, par la structure du feedback).
